In [ ]:
# 1. 필수 라이브러리 설치
!pip install torch transformers pandas scikit-learn
!pip install 'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer&subdirectory=kobert_hf'

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 2. 데이터 로드 (Colab에 train_data.csv 업로드 필수)
try:
    df = pd.read_csv('train_data.csv', encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv('train_data.csv', encoding='euc-kr')

# 데이터 전처리 (결측치 제거)
df = df.dropna(subset=['title', 'topic_idx']) # 가짜뉴스 데이터라면 'title', 'label' 등으로 수정

# [중요] 빠른 실습을 위해 10,000개만 샘플링 (전체 데이터 사용 시 이 줄 주석 처리)
df = df.sample(n=10000, random_state=42)

# 입력(X)과 정답(y) 분리
X = df['title']      # 뉴스 제목
y = df['topic_idx']  # 정답 레이블 (가짜뉴스: 0/1, 뉴스토픽: 0~6)

# 학습용/테스트용 데이터 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"데이터 준비 완료! 학습 데이터: {len(X_train)}개, 테스트 데이터: {len(X_test)}개")

  Cloning https://github.com/SKTBrain/KoBERT.git to /tmp/pip-install-nq6bahwo/kobert-tokenizer_389a7b0026a6401f843df084531eff30
  Running command git clone --filter=blob:none --quiet https://github.com/SKTBrain/KoBERT.git /tmp/pip-install-nq6bahwo/kobert-tokenizer_389a7b0026a6401f843df084531eff30
  Resolved https://github.com/SKTBrain/KoBERT.git to commit fcd729f2f4b37858f333597c0782388ada51eb5f
  Preparing metadata (setup.py) ... done
데이터 준비 완료! 학습 데이터: 8000개, 테스트 데이터: 2000개


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("=== [1] 베이스라인 모델 학습 시작 ===")

# 1. TF-IDF 벡터화 (텍스트 -> 숫자 변환)
vectorizer = TfidfVectorizer(max_features=5000) # 상위 5000개 단어만 사용
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 2. 로지스틱 회귀 모델 학습
baseline_model = LogisticRegression(max_iter=1000, n_jobs=-1)
baseline_model.fit(X_train_vec, y_train)

# 3. 예측 및 평가
y_pred_base = baseline_model.predict(X_test_vec)
acc_base = accuracy_score(y_test, y_pred_base)

print(f"\n>> 베이스라인(TF-IDF) 정확도: {acc_base * 100:.2f}%")
print("=====================================")

=== [1] 베이스라인 모델 학습 시작 ===

>> 베이스라인(TF-IDF) 정확도: 71.30%


In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel
from torch.optim import AdamW  # [수정됨] transformers가 아닌 torch.optim에서 가져옴
from kobert_tokenizer import KoBERTTokenizer
from tqdm.notebook import tqdm

print("=== [2] KoBERT 모델 학습 시작 ===")

# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

# 1. 하이퍼파라미터 설정
MAX_LEN = 64
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 2e-5
NUM_LABELS = len(y.unique()) # 레이블 개수 자동 인식

# 2. 토크나이저 및 데이터셋 클래스
tokenizer = KoBERTTokenizer.from_pretrained('skt/kobert-base-v1')

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 데이터로더 생성
train_ds = NewsDataset(X_train, y_train, tokenizer, MAX_LEN)
test_ds = NewsDataset(X_test, y_test, tokenizer, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# 3. KoBERT 분류 모델 정의
class KoBERTClassifier(nn.Module):
    def __init__(self, num_labels):
        super(KoBERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained('skt/kobert-base-v1')
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, num_labels) # 768 -> 레이블 개수

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        x = self.dropout(pooled_output)
        logits = self.classifier(x)
        return logits

model = KoBERTClassifier(num_labels=NUM_LABELS).to(device)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()

# 4. 학습 함수
def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# 5. 평가 함수
def eval_epoch(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            _, preds = torch.max(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

# 6. 실제 학습 실행
for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    train_loss = train_epoch(model, train_loader, optimizer, loss_fn)
    val_acc = eval_epoch(model, test_loader)
    print(f"Loss: {train_loss:.4f}, Accuracy: {val_acc*100:.2f}%")

print(f"\n>> KoBERT 최종 정확도: {val_acc * 100:.2f}%")
print("=====================================")

=== [2] KoBERT 모델 학습 시작 ===
사용 장치: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

...

Evaluating:   0%|          | 0/63 [00:00<?, ?it/s]

Loss: 0.3620, Accuracy: 87.05%
Epoch 3/3


Training:   0%|          | 0/250 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/63 [00:00<?, ?it/s]

Loss: 0.2392, Accuracy: 87.85%

>> KoBERT 최종 정확도: 87.85%


In [ ]:
# 7. 실제 사용 테스트 (Demo)
def predict_news(input_text):
    model.eval()
    encoding = tokenizer.encode_plus(
        input_text,
        add_special_tokens=True,
        max_length=64,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
        _, preds = torch.max(outputs, dim=1)

    # 레이블 맵핑 (여러분의 데이터에 맞게 수정 필요)
    # 가짜뉴스(0)/진짜뉴스(1)인 경우
    result = "가짜 뉴스(Fake)" if preds.item() == 0 else "진짜 뉴스(Real)"

    # DACON 뉴스토픽인 경우 (예시)
    topic_map = {0:'IT  과학', 1:'경제', 2:'사회', 3:'생활/문화', 4:'세계', 5:'스포츠', 6:'정치'}
    if NUM_LABELS > 2:
        result = topic_map.get(preds.item(), "알수없음")

    print(f"입력 기사: {input_text}")
    print(f"모델 예측: {result}")
    print("-" * 30)

# 테스트 실행
print("\n=== [Demo] 모델 테스트 ===")
predict_news("반도체 수출 역대 최고치 달성, 경제 회복 신호탄인가")
predict_news("손흥민 리그 10호골 폭발, 토트넘 승리 견인")
predict_news("정부, 내년부터 새로운 부동산 정책 시행 발표")


=== [Demo] 모델 테스트 ===
입력 기사: 반도체 수출 역대 최고치 달성, 경제 회복 신호탄인가
모델 예측: 경제
------------------------------
입력 기사: 손흥민 리그 10호골 폭발, 토트넘 승리 견인
모델 예측: 스포츠
------------------------------
입력 기사: 정부, 내년부터 새로운 부동산 정책 시행 발표
모델 예측: 정치
------------------------------
